# SafeRide Vision — Google Colab

This notebook is the supported way to run the project. It clones the code from GitHub, installs only missing runtime packages, loads every model weight from Google Drive or a Colab upload, then starts the API.

Before running, set `REPO_URL` below. Put weights in `MyDrive/Motor-Bikes-Data/local_project_weights` or upload the missing files when prompted.

In [ ]:
# GitHub repository containing this project
REPO_URL = "https://github.com/Salal04/Safe-Ride.git"  # replace this
BRANCH = "main"
REPO_DIR = "/content/Safe-Ride"

import subprocess
from pathlib import Path
if "YOUR_ACCOUNT" in REPO_URL or "YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Replace REPO_URL with your GitHub repository URL.")
repo = Path(REPO_DIR)
if (repo / ".git").exists():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists and is not a Git checkout.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print("Repository ready:", REPO_DIR)

In [ ]:
# Mount Google Drive. Colab already includes PyTorch and common scientific packages.
from google.colab import drive
drive.mount("/content/drive")
%cd /content/Safe-Ride

# Install only missing runtime packages. Avoid pinning/reinstalling Colab's torch/CUDA stack.
import importlib.util
import subprocess
import sys

runtime_packages = {
    'ultralytics': 'ultralytics',
    'flask': 'Flask',
    'flask_cors': 'Flask-Cors',
    'pyngrok': 'pyngrok',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'tiktoken': 'tiktoken',
    'verovio': 'verovio',
}
missing_packages = [package for module, package in runtime_packages.items()
                   if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages only:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All application runtime packages are already installed.')

# Confirm Colab's existing core ML stack; don't replace it.
for module in ('torch', 'torchvision', 'cv2', 'numpy', 'scipy'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Required Colab package is missing: {module}')
print('Using existing Colab PyTorch/CUDA stack.')

## Load model weights

All model paths and filenames come from `config.py`. Checkpoints are read from `MyDrive/Motor-Bikes-Data/local_project_weights`. Copy them there before running, or select all missing files in the upload dialog. Upload filenames must match the configured basenames.

In [ ]:
import sys
from pathlib import Path
from google.colab import files

sys.path.insert(0, REPO_DIR)
import config

required = config.REQUIRED_WEIGHTS
missing = [(name, Path(path)) for name, path in required.items() if not Path(path).is_file()]
if missing:
    print("Upload these missing weight files:")
    for name, path in missing:
        print(f"  {name}: {path.name}")
    uploaded = files.upload()
    for name, destination in missing:
        filename = destination.name
        if filename not in uploaded:
            raise FileNotFoundError(f"Upload the configured filename exactly: {filename}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(uploaded[filename])
        print(f"Saved {name} to {destination}")

for name, path in required.items():
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty model weight '{name}': {path}")
    print(f"Ready: {name} -> {path} ({path.stat().st_size:,} bytes)")

## Start the backend

A valid ngrok authtoken is required to expose the backend from Colab. Enter it privately when prompted; it is passed to the backend process through a temporary environment variable and is not saved in the notebook.

In [ ]:
# Enter the valid authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
from getpass import getpass
import os
from pyngrok import conf
token = "3HPHJOJp5DllsaDIgvKojBfKlZ0_2J5irPkTDFdEzg3g7tHyM"
if not token:
    raise ValueError("A valid ngrok authtoken is required to start the public Colab tunnel.")
os.environ['NGROK_AUTHTOKEN'] = token

%cd /content/Safe-Ride
!python backend_server.py